In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    set_seed,
)
import datasets
import warnings
from luh import AutoUncertaintyHead

from transformers.modeling_utils import PreTrainedModel, PretrainedConfig
from transformers.modeling_outputs import CausalLMOutput
import torch
from itertools import chain
from typing import Optional
from dataclasses import dataclass

from scipy.special import expit
import numpy as np
import pandas as pd

/home/jovyan/gkuzmin/llm_ue/conda_llm_ue_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass
class CausalLMWithUncertaintyOutput(CausalLMOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    uncertainty: torch.FloatTensor = None


class CausalLMWithUncertaintyLayerClaim(PreTrainedModel):
    def __init__(
        self,
        base_model,
        ue_head,
        ue_pos_weight: float,
        output_attention: bool = False
    ):
        super().__init__(PretrainedConfig())

        self.orig_base_model = base_model
        self.ue_head = ue_head
        self._output_attention = output_attention
        self._ue_pos_weight = ue_pos_weight

    def generate(self, *args, **kwargs):
        raise NotImplementedError

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        context_lengths=None,
        claims=None,
        verified=None,
        return_dict=None,
        **kwargs
    ):
        return_dict = (
            return_dict
            if return_dict is not None
            else self.orig_base_model.config.use_return_dict
        )
        output_hidden_states = True
        output_attentions = self._output_attention

        outputs = self.orig_base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            **kwargs
        )
        outputs.logits = outputs.logits.float()
        logits = outputs.logits
        outputs.context_lengths = context_lengths

        ue_head_input = {"input_ids": input_ids, 
                         "attention_mask": attention_mask,
                         "claims": claims}

        uncertainty = self.ue_head(ue_head_input, outputs)
        loss = None

        return CausalLMWithUncertaintyOutput(
            loss=loss,
            logits=logits,
            uncertainty=uncertainty,
        )

In [3]:
def load_tokenizer(pretrained_model_name_or_path):
    padding_side = "right"
    tokenizer = AutoTokenizer.from_pretrained(
        pretrained_model_name_or_path,
        model_max_length=2400,
        padding_side=padding_side,
        cache_dir=None,
        token=None,
    )
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model(pretrained_model_name_or_path, ue_layer_path, pos_weight=5.0, device_map="cuda"):
    base_model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map=device_map,
        cache_dir=None,
        token=None,
    )

    uq_head = AutoUncertaintyHead.from_pretrained(ue_layer_path, base_model)

    if uq_head.model_type == "claim":
        model = CausalLMWithUncertaintyLayerClaim(
            base_model,
            ue_head=uq_head,
            ue_pos_weight=pos_weight,
            output_attention=uq_head.output_attentions,
        )
        model.to(device_map)
    for name, param in model.named_parameters():
        # use only for infernce
        param.requires_grad = False
    return model


In [30]:
class DataCollatorForLanguageModelingWithUncertaintyClaim(DataCollatorForLanguageModeling):
    def __init__(self, tokenizer, label_name="verified", *args, **kwargs):
        self._tokenizer = tokenizer
        self.label_name = label_name
        super().__init__(tokenizer, *args, **kwargs)

    def _adjust_claim_positions(self, context_length, input_ids, claim_obj):
        claim_token_positions = claim_obj['aligned_token_ids']
        mapping = []
        for idx, token_id in enumerate(input_ids[context_length:]):
            if token_id not in self.tokenizer.all_special_ids:
                mapping.append(idx)
        if len(mapping) != len(claim_token_positions):
            warnings.warn("Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered")
            # in case then we have special_ids in claim due to custom claim splitting
            for idx, token_id in enumerate(input_ids[context_length:]):
                if token_id in self.tokenizer.all_special_ids and idx < len(claim_token_positions):
                    del claim_token_positions[idx]
        adjusted_positions = [mapping[i] for i in claim_token_positions]
        return context_length + torch.tensor(adjusted_positions)
    

    def torch_call(self, examples):
        batch_size = len(examples)
        # Do padding
        batch = super().torch_call([{'input_ids' : e['input_ids']} for e in examples]) 
        # Construct context lengths
        context_lengths = []
        for i in range(batch_size):
            reply_len = len(examples[i]['input_ids']) - len(examples[i]['prompt_tokens'])
            if self._tokenizer.padding_side == "right":
                # in case of non-inst giga - remove 1 (eos)
                tok_diff = 2 if "giga" in self._tokenizer.name_or_path else 0
                context_lengths.append(len(examples[i]['prompt_tokens'])-tok_diff)
            else:
                context_lengths.append(batch["input_ids"][i].shape[0] - reply_len)
        
        batch["context_lengths"] = torch.tensor(context_lengths)
        # Construct claim tensors
        all_claim_tensors = []
        for i in range(len(batch["input_ids"])):
            instance_claims = []
            for claim in examples[i]["claims"]:
                mask = torch.zeros(batch["input_ids"][i].shape, dtype=int)
                claim_token_positions = self._adjust_claim_positions(batch["context_lengths"][i], batch["input_ids"][i], claim)
                if any(e > mask.shape[0] for e in claim_token_positions):
                    print("Error")
                
                mask[claim_token_positions] = 1
                instance_claims.append(mask[1:]) # ignoring <s>

            all_claim_tensors.append(torch.stack(instance_claims) if len(instance_claims) > 0 else torch.zeros(0, full_len - 1, dtype=int))

        batch["claims"] = all_claim_tensors
        dict_batch = dict(batch)
        return dict_batch

In [31]:
class UHeadClaimEvaluator():
    def __init__(self, model_name, uhead_name, device, batch_size=8):
        self.uhead = load_model(model_name, uhead_name, device_map=device)
        self.uhead.eval()
        self.device = device
        self.tokenizer = load_tokenizer(model_name)
        self.collator = DataCollatorForLanguageModelingWithUncertaintyClaim(self.tokenizer, mlm=False)
        self.batch_size = batch_size
        self.thresholds = {0.3083: "low", 0.7: "medium", 1.0: "high"}
        self.thresholds_int = {-1: "low", 0: "medium", 1: "high"}
        self.thr_vectorized = np.vectorize(lambda x: self.thresholds_int[x])

    def prepare_inputs(self, prompts, outputs, claims):
        assert len(prompts) == len(outputs)
        input_dict = {}
        input_dict["question"] = prompts
        input_dict["reply"] = outputs
        # to add - input_ids, greedy_ids
        # TODO: add batched version
        input_dict["input_ids"] = [self.tokenizer(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(input_dict["question"], input_dict["reply"])]
        input_dict["greedy_ids"] = [self.tokenizer(str(el2))["input_ids"][1:-1] for el2 in input_dict["reply"]]
        input_dict["prompt_tokens"] = [self.tokenizer(el1, padding=False, return_tensors=None)["input_ids"] for el1 in input_dict["question"]]
        # now process and add claims
        input_dict["claims"] = []
        for idx, el in enumerate(input_dict["greedy_ids"]):
            buf_claim_data = []
            buf_claims = claims[idx]
            prev_matched_tok_start = 0
            for inner_idx, text_claim in enumerate(buf_claims):
                # strip bos and eos
                claim_tokens = None
                claim_tokens = self.tokenizer(str(text_claim))["input_ids"][1:-1]
                matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
                if len(matched_toks) == 0:
                    print(f"Claim {text_claim} for sample {idx} is not matched!")
                else:
                    matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
                    prev_matched_tok_start = matched_toks[0]+1
                    buf_claim_data.append({"aligned_token_ids": matched_toks,
                                           "claim_text": str(text_claim),
                                           "sentence": input_dict["reply"][idx]})
            input_dict["claims"].append(buf_claim_data)
        return datasets.Dataset.from_dict(input_dict)

    def _threshold_scores(self, ue_scores):
        thresholds_keys = list(self.thresholds.keys())
        ue_scores = np.where(ue_scores <= thresholds_keys[0], -1, ue_scores)
        ue_scores = np.where(np.logical_and(ue_scores > thresholds_keys[0], ue_scores <= thresholds_keys[1]), 0, ue_scores)
        ue_scores = np.where(np.logical_and(ue_scores > thresholds_keys[1], ue_scores <= thresholds_keys[2]), 1, ue_scores)
        return self.thr_vectorized(ue_scores)

    def __call__old(self, prompts, outputs, claims, use_threshold=False):
        # preprocess and prepare all inputs
        samples_for_inference = self.prepare_inputs(prompts, outputs, claims)
        # collate samples
        # TODO: add batched inference
        samples_for_inference = self.collator.torch_call(samples_for_inference)
        for key in samples_for_inference.keys():
            if isinstance(samples_for_inference[key], torch.Tensor):
                samples_for_inference[key] = samples_for_inference[key].to(self.device)
            if isinstance(samples_for_inference[key], list):
                samples_for_inference[key] = [el.to(self.device) for el in samples_for_inference[key]]
        # predict
        with torch.no_grad():
            preds = self.uhead(**samples_for_inference)
        # postprocess scores
        ue_scores = 1 - expit(preds.uncertainty.detach().cpu().numpy())
        if use_threshold:
            ue_scores = self._threshold_scores(ue_scores)
        return ue_scores.tolist()

    def __call__(self, prompts, outputs, claims, use_threshold=False, batch_size=None):
        # preprocess all samples
        samples = self.prepare_inputs(prompts, outputs, claims)
        batch_size = min(self.batch_size, len(samples))
    
        ue_scores_all = []
        with torch.no_grad():
            for start in range(0, len(samples), batch_size):
                indices = np.arange(start, min(start + batch_size, len(samples)))
                print(indices)
                batch = samples.select(indices)
                # collate batch
                batch = self.collator.torch_call(batch)
                # move to device
                for key, value in batch.items():
                    if isinstance(value, torch.Tensor):
                        batch[key] = value.to(self.device)
                    elif isinstance(value, list):
                        batch[key] = [el.to(self.device) if hasattr(el, "to") else el for el in value]
                # predict
                preds = self.uhead(**batch)
                # postprocess scores
                ue_scores = 1 - expit(preds.uncertainty.detach().cpu().numpy())
                ue_scores_all.append(ue_scores)
        ue_scores_all = np.concatenate(ue_scores_all, axis=0)
        if use_threshold:
            ue_scores_all = self._threshold_scores(ue_scores_all)
        return ue_scores_all.tolist()
        


In [32]:

def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key] = df.iloc[idx][key]
        results.append(buf_res)
    return results
preds_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim/train/2025-12-22/18-38-06/predictions/"
exp1_uhead_scores = datasets.load_from_disk(preds_path)
exp1_uhead_scores = np.array(exp1_uhead_scores['uncertainty_logits'])
exp1_uhead_scores = expit(exp1_uhead_scores)#expit(exp1_uhead_scores[exp1_uhead_scores != -100])
exp1_uhead_scores_flat = expit(exp1_uhead_scores[exp1_uhead_scores != -100])
path = "../med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv"
df = pd.read_csv(path)
results = results_from_csv(df)
test_ids = list(range(len(df)))
results = [results[idx] for idx in test_ids]
# for claim level - 
TARGET_METRIC = 'filtered_claim_is_true'
metric_df = pd.read_csv("../med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv")
metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]

In [33]:
# input example
test_idx = 0
indices = [0, 1]
#indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
prompts, outputs, claims = [], [], []
for test_idx in indices:
    prompt, output, claims_raw = results[test_idx]["input"], results[test_idx]["prediction"], results[test_idx]["estimations"]["claims"]
    buf_claims = [str(el.strip("'")) for el in claims_raw.strip('[').strip(']').split(", ")]
    prompts.append(prompt)
    outputs.append(output)
    claims.append(buf_claims)

In [34]:
model_name = "/home/jovyan/rvanova/giga1/Lite_v2/hf"
uhead_name = "./DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim/train/2025-12-18/01-26-09/model"
device = "cuda:0"
evaluator = UHeadClaimEvaluator(model_name, uhead_name, device)

Loading checkpoint shards: 100%|██████████| 8/8 [00:06<00:00,  1.16it/s]


In [35]:
new_dataset_v1 = datasets.load_from_disk("../newly_builded_datasets/med_data/gigachat_big/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set")
buf_example = new_dataset_v1[0]

In [36]:
buf_prep = evaluator.prepare_inputs(prompts, outputs, claims)

In [37]:
evaluator.tokenizer.all_special_ids

[1, 2, 0]

In [38]:
evaluator(prompts, outputs, claims, use_threshold=False)

[0 1]


/tmp/ipykernel_3207860/3817144487.py:14: UserWarning: Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered
  warnings.warn("Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered")


[[[0.7042843103408813], [0.07534104585647583], [0.05332469940185547]],
 [[0.41306841373443604], [0.01999688148498535], [0.20312947034835815]]]

In [12]:
buf_prep[0]

{'question': 'Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 45 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>\n<врач> Что вас беспокоит?\n<пациентка> боль в левом подреберье\n<врач> Есть ли у вас наследственная предрасположенность к заболеваниям ЖКТ?\n<пациентка> гастрит\n<врач> Принимаете ли какие-либо лекарства для облегчения симптомов?\n<пациентка> ношпа\n<врач> Имеет ли боль постоянный характер или она возникает периодически?\n<пациентка> периодически\n<врач> Испытываете ли вы чувство тяжести в животе после приема пищи?\n<пациентка> да\n<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском\n<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз\n</диалог>\nНе объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами\nКаждый ответ оберни в теги <diag_diag> и </diag_diag>',
 

In [10]:
buf_example

{'question': 'Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 45 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>\n<врач> Что вас беспокоит?\n<пациентка> боль в левом подреберье\n<врач> Есть ли у вас наследственная предрасположенность к заболеваниям ЖКТ?\n<пациентка> гастрит\n<врач> Принимаете ли какие-либо лекарства для облегчения симптомов?\n<пациентка> ношпа\n<врач> Имеет ли боль постоянный характер или она возникает периодически?\n<пациентка> периодически\n<врач> Испытываете ли вы чувство тяжести в животе после приема пищи?\n<пациентка> да\n<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском\n<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз\n</диалог>\nНе объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами\nКаждый ответ оберни в теги <diag_diag> и </diag_diag>',
 

In [28]:
evaluator(prompts, outputs, claims, use_threshold=False)

[0 1 2 3 4 5 6 7]


/tmp/ipykernel_3187642/2611082036.py:14: UserWarning: Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered
  warnings.warn("Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered")


[8 9]


[[[0.2178187519311905], [0.8276309967041016], [0.9550320506095886]],
 [[0.4759988784790039], [0.9246112704277039], [0.6967487335205078]],
 [[0.5896039605140686], [0.8876055479049683], [0.9143412113189697]],
 [[0.348450630903244], [0.3179451525211334], [0.841438353061676]],
 [[0.47847780585289], [0.6463838219642639], [0.9180791974067688]],
 [[0.9490369558334351], [0.8142881989479065], [0.9143186211585999]],
 [[0.43885838985443115], [0.8148345947265625], [0.9701043963432312]],
 [[0.5969954133033752], [0.880342423915863], [0.8986878991127014]],
 [[0.8939935564994812], [0.9279375076293945], [0.9752580523490906]],
 [[0.8055540919303894], [0.9568681120872498], [0.9779652953147888]]]

In [29]:
evaluator(prompts, outputs, claims, use_threshold=False)

/tmp/ipykernel_3184919/2611082036.py:14: UserWarning: Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered
  warnings.warn("Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered")


[[[0.2172614336013794], [0.8275560736656189], [0.9548994302749634]],
 [[0.4746103584766388], [0.9245286583900452], [0.6974132657051086]]]

In [30]:
evaluator(prompts, outputs, claims, use_threshold=True)

/tmp/ipykernel_3184919/2611082036.py:14: UserWarning: Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered
  warnings.warn("Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered")


[[['low'], ['high'], ['high']], [['medium'], ['high'], ['medium']]]

In [ ]:
def prompt_tokens(inst):
    return {"prompt_tokens": tokenizer(inst["question"], padding=False, return_tensors=None)["input_ids"]}

In [ ]:
model_name = "/home/jovyan/rvanova/giga1/Lite_v2/hf"
uhead_name = "./DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim/train/2025-12-18/01-26-09/model"
device = "cuda:0"
uhead = load_model(model_name, uhead_name, device_map=device)
tokenizer = load_tokenizer(model_name)

Loading checkpoint shards: 100%|██████████| 8/8 [00:06<00:00,  1.15it/s]


In [ ]:

def results_from_csv(df):
    results = []
    for idx in range(len(df)):
        buf_res = {}
        buf_res["input"] = df.iloc[idx]["input"]
        buf_res["target"] = df.iloc[idx]["target"]
        buf_res["prediction"] = df.iloc[idx]["prediction"]
        buf_res["estimations"] = {}
        for key in df.columns:
            if key not in ["input", "target", "prediction"]:
                buf_res["estimations"][key] = df.iloc[idx][key]
        results.append(buf_res)
    return results
preds_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim/train/2025-12-22/18-38-06/predictions/"
exp1_uhead_scores = datasets.load_from_disk(preds_path)
exp1_uhead_scores = np.array(exp1_uhead_scores['uncertainty_logits'])
exp1_uhead_scores = expit(exp1_uhead_scores)#expit(exp1_uhead_scores[exp1_uhead_scores != -100])
exp1_uhead_scores_flat = expit(exp1_uhead_scores[exp1_uhead_scores != -100])
path = "../med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv"
df = pd.read_csv(path)
results = results_from_csv(df)
test_ids = list(range(len(df)))
results = [results[idx] for idx in test_ids]
# for claim level - 
TARGET_METRIC = 'filtered_claim_is_true'
metric_df = pd.read_csv("../med_data/medical_data_ordered_giga_big_fixed_format_holdout_set_with_claims.csv")
metric_scores = [list(map(int, metric_df[TARGET_METRIC][idx].replace("[","").replace("]","").replace(",","").split())) for idx in test_ids]

In [ ]:
# input example
test_idx = 0
indices = [0, 1]
prompts, outputs, claims = [], [], []
for test_idx in indices:
    prompt, output, claims_raw = results[test_idx]["input"], results[test_idx]["prediction"], results[test_idx]["estimations"]["claims"]
    buf_claims = [str(el.strip("'")) for el in claims_raw.strip('[').strip(']').split(", ")]
    prompts.append(prompt)
    outputs.append(output)
    claims.append(buf_claims)

In [ ]:
def prepare_inputs(prompts, outputs, claims, tokenizer):
    assert len(prompts) == len(outputs)
    input_dict = {}
    input_dict["question"] = prompts
    input_dict["reply"] = outputs
    # to add - input_ids, greedy_ids
    # TODO: add batched version
    input_dict["input_ids"] = [tokenizer(el1 + str(el2))["input_ids"][:-1] for el1, el2 in zip(input_dict["question"], input_dict["reply"])]
    input_dict["greedy_ids"] = [tokenizer(str(el2))["input_ids"][1:-1] for el2 in input_dict["reply"]]
    input_dict["prompt_tokens"] = [tokenizer(el1, padding=False, return_tensors=None)["input_ids"] for el1 in input_dict["question"]]
    # now process and add claims
    input_dict["claims"] = []
    for idx, el in enumerate(input_dict["greedy_ids"]):
        buf_claim_data = []
        buf_claims = claims[idx]
        prev_matched_tok_start = 0
        for inner_idx, text_claim in enumerate(buf_claims):
            # strip bos and eos
            claim_tokens = None
            claim_tokens = tokenizer(str(text_claim))["input_ids"][1:-1]
            matched_toks = [x for x in range(len(el))[prev_matched_tok_start:] if el[x:x+len(claim_tokens)] == claim_tokens]
            if len(matched_toks) == 0:
                print(f"Claim {text_claim} for sample {idx} is not matched!")
            else:
                matched_toks = [x + matched_toks[0] for x in range(len(claim_tokens))]
                prev_matched_tok_start = matched_toks[0]+1
                buf_claim_data.append({"aligned_token_ids": matched_toks,
                                       "claim_text": str(text_claim),
                                       "sentence": input_dict["reply"][idx]})
        input_dict["claims"].append(buf_claim_data)
    return datasets.Dataset.from_dict(input_dict)

In [ ]:
test_inference_from_raw = prepare_inputs(prompts, outputs, claims, tokenizer)

In [ ]:
# do inference
collator = DataCollatorForLanguageModelingWithUncertaintyClaim(tokenizer, mlm=False)
test_inference_from_raw = collator.torch_call(test_inference_from_raw)
for key in test_inference_from_raw.keys():
    if isinstance(test_inference_from_raw[key], torch.Tensor):
        test_inference_from_raw[key] = test_inference_from_raw[key].to(device)
    if isinstance(test_inference_from_raw[key], list):
        test_inference_from_raw[key] = [el.to(device) for el in test_inference_from_raw[key]]

/tmp/ipykernel_3184035/2611082036.py:14: UserWarning: Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered
  warnings.warn("Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered")


In [ ]:
uhead.eval()

CausalLMWithUncertaintyLayerClaim(
  (orig_base_model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(128256, 3584, padding_idx=2)
      (layers): ModuleList(
        (0-34): 35 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=3584, out_features=3584, bias=False)
            (k_proj): Linear(in_features=3584, out_features=512, bias=False)
            (v_proj): Linear(in_features=3584, out_features=512, bias=False)
            (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
            (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
            (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): LlamaRMSNorm((3584,), eps=1e-06)
          (post_attention_layernorm):

In [ ]:
with torch.no_grad():
    preds = uhead(**test_inference_from_raw)

torch.float32
torch.float32


In [15]:
expit(preds.uncertainty.detach().cpu().numpy())

array([[[0.21726143],
        [0.8275561 ],
        [0.95489943]],

       [[0.47461036],
        [0.92452866],
        [0.69741327]]], dtype=float32)

In [98]:
exp1_uhead_scores[:2]

array([[[0.21783628],
        [0.82768892],
        [0.95506836]],

       [[0.47606202],
        [0.92468769],
        [0.69678426]]])

In [57]:
t1 = expit(preds.uncertainty.detach().cpu().numpy())

In [58]:
t1

array([[[0.21726143],
        [0.8275561 ],
        [0.95489943]],

       [[0.47461036],
        [0.92452866],
        [0.69741327]]], dtype=float32)

In [81]:
thresholds = {0.3083: "low", 0.7: "medium", 1.0: "high"}
thresholds_int = {-1: "low", 0: "medium", 1: "high"}
thresholds_keys = list(thresholds.keys())
thr = lambda x: thresholds_int[x]

In [83]:
t1 = np.where(t1 <= thresholds_keys[0], -1, t1)
t1 = np.where(np.logical_and(t1 > thresholds_keys[0], t1 <= thresholds_keys[1]), 0, t1)
t1 = np.where(np.logical_and(t1 > thresholds_keys[1], t1 <= thresholds_keys[2]), 1, t1)

#t1 = np.where(t1>0.3083, 2, t1)

In [84]:
thr_vectorized = np.vectorize(thr)

In [85]:
t1

array([[[-1.],
        [ 1.],
        [ 1.]],

       [[ 0.],
        [ 1.],
        [ 0.]]], dtype=float32)

In [86]:
thr_vectorized(t1)
#np.apply_along_axis(thr, 1, t1)

array([[['low'],
        ['high'],
        ['high']],

       [['medium'],
        ['high'],
        ['medium']]], dtype='<U6')

In [61]:
t1

array([[[0.3083],
        [1.    ],
        [1.    ]],

       [[0.7   ],
        [1.    ],
        [0.7   ]]], dtype=float32)

In [62]:
prompt, output, claims

('Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 45 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>\n<врач> Что вас беспокоит?\n<пациентка> боль в левом подреберье\n<врач> Есть ли у вас наследственная предрасположенность к заболеваниям ЖКТ?\n<пациентка> гастрит\n<врач> Принимаете ли какие-либо лекарства для облегчения симптомов?\n<пациентка> ношпа\n<врач> Имеет ли боль постоянный характер или она возникает периодически?\n<пациентка> периодически\n<врач> Испытываете ли вы чувство тяжести в животе после приема пищи?\n<пациентка> да\n<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском\n<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз\n</диалог>\nНе объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами\nКаждый ответ оберни в теги <diag_diag> и </diag_diag>',
 '<diag_diag>

In [63]:
# to add - input_ids, greedy_ids, claims

In [64]:
some_ids = [0, 1]
new_dataset_v1 = datasets.load_from_disk("../newly_builded_datasets/med_data/gigachat_big/raw_med_data_giga_big_relaxed_em_target_claim_fixed_format_holdout_set")
buf_example = new_dataset_v1[test_idx]

In [65]:
buf_example

{'question': 'Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 45 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>\n<врач> Что вас беспокоит?\n<пациентка> боль в левом подреберье\n<врач> Есть ли у вас наследственная предрасположенность к заболеваниям ЖКТ?\n<пациентка> гастрит\n<врач> Принимаете ли какие-либо лекарства для облегчения симптомов?\n<пациентка> ношпа\n<врач> Имеет ли боль постоянный характер или она возникает периодически?\n<пациентка> периодически\n<врач> Испытываете ли вы чувство тяжести в животе после приема пищи?\n<пациентка> да\n<пациентка> Не объясняй их, просто напиши 3 самых вероятных потенциальных диагноза на русском\n<пациентка> Не пиши пояснений к диагнозу в скобках, просто напиши самое подходящие название диагноза один раз\n</диалог>\nНе объясняй их, просто напиши 3 самых вероятных потенциальных диагноза обернув тегами\nКаждый ответ оберни в теги <diag_diag> и </diag_diag>',
 

In [86]:
new_dataset_v1[1]

{'question': 'Ты врач с большим опытом работы.\nЯ пациент женского пола. Мне 45 лет. Я обратилась к тебе с проблемой.\nНазови 3 самых вероятных потенциальных диагноза, основываясь на диалоге:\n<диалог>\n<врач> Как давно появились жалобы?\n<пациентка> позавчера\n<врач> Правильно я понимаю, что симптомы двусторонние?\n<пациентка> да\n<врач> Есть ли проблемы со зрением?\n<пациентка> нет, только долго не могу с открытыми глазами, хочется закрыть глаза\n<врач> Беспокоит ли отделяемое из глаз, если да, какого цвета\n<пациентка> прозрачное, достаточно обильное\n<врач> Помимо жалоб со стороны глаз, есть ли другие симптомы?\n<пациентка> Знобит, ломит тело\n<врач> Повышается ли температура тела?\n<пациентка> не мерил\n<врач> Был ли контакт с инфекционными больными?\n<пациентка> дома дети болеют, ОРВИ\n<врач> Заложенность носа, боль в горле беспокоят?\n<пациентка> горло немного першит и всё\n<врач> Есть ли высыпания на веках, коже вокруг глаз?\n<пациентка> нет, ничего\n<врач> Поняла, сейчас задам

In [66]:
buf_example.keys()

dict_keys(['question', 'answer', 'reply', 'input_ids', 'greedy_ids', 'claims', 'verified', 'uncertainty_labels', 'uncertainty_labels_original'])

In [67]:
# do inference
collator = DataCollatorForLanguageModelingWithUncertaintyClaim(tokenizer, mlm=False)
inference_sample = datasets.Dataset.from_dict(new_dataset_v1[some_ids])
inference_sample = inference_sample.map(prompt_tokens)

Map: 100%|██████████| 2/2 [00:00<00:00, 209.40 examples/s]


In [68]:
inference_sample = collator.torch_call(inference_sample)

/tmp/ipykernel_3181013/2901398519.py:21: UserWarning: Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered
  warnings.warn("Number of aligned tokens does not match number of generated tokens in claim; please check if special_ids filtered")


In [69]:
for key in inference_sample.keys():
    if isinstance(inference_sample[key], torch.Tensor):
        inference_sample[key] = inference_sample[key].to(device)
    if isinstance(inference_sample[key], list):
        inference_sample[key] = [el.to(device) for el in inference_sample[key]]

In [70]:
inference_sample

{'input_ids': tensor([[     1,  52726,  11391,    324,  21817,  38289,   4499,    332,   2559,
           22436,  60897,  22296,     16,  29809,    223,     22,     23,   3097,
              16,   3508,  67983,    355,  20576,    324,  42827,    332,  37612,
             342,    262,    223,     21,  10078,  16110,    751,  72050,  82981,
              14,  59687,  15860,    438,  36492,    261,    623,     30,   1708,
            6873,    756,     30,    279,  17820,     32,  13621,   5915, 105448,
            1486,     30,   3651,    836,   1042,    647,     32,   1960,    322,
           62257,    946,   2307,    361,   5919,    201,     30,    279,  17820,
              32,  21743,   1320,    487,   5915,  15947,  15470, 105975,  11975,
             355, 106040,  86641,   1486,     30,   3651,    836,   1042,    647,
              32,  33827,   2728,    201,     30,    279,  17820,     32,   3927,
            1676,  12580,   1320,  13164,  14396,  41216,   1030, 108852,  42536,
   

In [71]:
uhead.eval()

CausalLMWithUncertaintyLayerClaim(
  (orig_base_model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(128256, 3584, padding_idx=2)
      (layers): ModuleList(
        (0-34): 35 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=3584, out_features=3584, bias=False)
            (k_proj): Linear(in_features=3584, out_features=512, bias=False)
            (v_proj): Linear(in_features=3584, out_features=512, bias=False)
            (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
            (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
            (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): LlamaRMSNorm((3584,), eps=1e-06)
          (post_attention_layernorm):

In [72]:
for key in inference_sample.keys():
    if isinstance(inference_sample[key], torch.Tensor):
        print(key, inference_sample[key].device)
    if isinstance(inference_sample[key], list):
        print(key, inference_sample[key][0].device)
        #inference_sample[key] = [el.to(device) for el in inference_sample[key]]

input_ids cuda:0
attention_mask cuda:0
labels cuda:0
context_lengths cuda:0
claims cuda:0


In [73]:
with torch.no_grad():
    preds = uhead(**inference_sample)

torch.float32
torch.float32
torch.float32
torch.float32
torch.bool
torch.float32


In [74]:
uhead.ue_head.classifier.bias.device

device(type='cuda', index=0)

In [76]:
preds.uncertainty

tensor([[[-1.2817],
         [ 1.5684],
         [ 3.0527]],

        [[-0.1016],
         [ 2.5055],
         [ 0.8350]]], device='cuda:0')

In [79]:
from scipy.special import expit
import numpy as np

In [80]:
preds_path = "/home/jovyan/gkuzmin/llm_ue/llm-uncertainty-layer/DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/holdout_evals/med_data_relaxed_em_target_claim/train/2025-12-22/18-38-06/predictions/"
exp1_uhead_scores = datasets.load_from_disk(preds_path)
exp1_uhead_scores = np.array(exp1_uhead_scores['uncertainty_logits'])
exp1_uhead_scores = expit(exp1_uhead_scores)#expit(exp1_uhead_scores[exp1_uhead_scores != -100])
exp1_uhead_scores_flat = expit(exp1_uhead_scores[exp1_uhead_scores != -100])

In [85]:
expit(preds.uncertainty.detach().cpu().numpy())

array([[[0.21726143],
        [0.8275561 ],
        [0.95489943]],

       [[0.47461036],
        [0.92452866],
        [0.69741327]]], dtype=float32)

In [81]:
exp1_uhead_scores

array([[[0.21783628],
        [0.82768892],
        [0.95506836]],

       [[0.47606202],
        [0.92468769],
        [0.69678426]],

       [[0.58959822],
        [0.88759491],
        [0.91429072]],

       [[0.34853426],
        [0.31795547],
        [0.84143515]],

       [[0.47846792],
        [0.64634861],
        [0.91804045]],

       [[0.94901185],
        [0.81434827],
        [0.91429072]],

       [[0.43887538],
        [0.81479079],
        [0.97012691]],

       [[0.59702066],
        [0.88038634],
        [0.89876652]],

       [[0.89423658],
        [0.92801837],
        [0.97534314]],

       [[0.80578506],
        [0.95711776],
        [0.97800978]],

       [[0.83073148],
        [0.78349191],
        [0.98184049]],

       [[0.33404611],
        [0.95405179],
        [0.96765567]],

       [[0.32071492],
        [0.60544918],
        [0.76734135]],

       [[0.51221223],
        [0.57832911],
        [0.97608361]],

       [[0.79085356],
        [0.79166005],
    

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key"

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from luh import AutoUncertaintyHead

from lm_polygraph import CausalLMWithUncertainty
from lm_polygraph.stat_calculators import ClaimsExtractor
from lm_polygraph.utils.openai_chat import OpenAIChat

from luh.calculator_infer_luh import CalculatorInferLuh
from luh.calculator_apply_uq_head import CalculatorApplyUQHead
from luh.luh_claim_estimator_dummy import LuhClaimEstimatorDummy

/home/jovyan/gkuzmin/llm_ue/conda_llm_ue_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load model and uhead
model_name = "/home/jovyan/rvanova/giga1/Lite_v2/hf"
uhead_name = "./DEBUG_workdir_med_data/gigachat_big/enlarged_train/fixed_format/med_data_relaxed_em_target_claim/train/2025-12-18/01-26-09/model"

llm = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="cuda", output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained(
    model_name)
tokenizer.pad_token = tokenizer.eos_token
uhead = AutoUncertaintyHead.from_pretrained(
    uhead_name, base_model=llm)

/home/jovyan/gkuzmin/llm_ue/conda_llm_ue_env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:820: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_attentions` is. When `return_dict_in_generate` is not `True`, `output_attentions` is ignored.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 8/8 [00:06<00:00,  1.19it/s]


In [4]:
generation_config = GenerationConfig.from_pretrained(model_name)
args_generate = {"generation_config": generation_config}
calc_infer_llm = CalculatorInferLuh(
    uhead,
    tokenize=True,
    args_generate=args_generate,
    device="cuda",
    generations_cache_dir="",
    predict_token_uncertainties=False,
)

openai_chat = OpenAIChat()
calc_steps_extractor = ClaimsExtractor(openai_chat)
calc_apply_uhead = CalculatorApplyUQHead(uhead)

estimator = LuhClaimEstimatorDummy()
llm_adapter = CausalLMWithUncertainty(
    llm,
    tokenizer=tokenizer,
    stat_calculators=[calc_infer_llm, calc_steps_extractor, calc_apply_uhead],
    estimator=estimator,
    keep_deps=["claims"]
)

In [5]:
# prepare text ...

messages = [
    [
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user", 
            "content": "In which year did the programming language Mercury first appear?"
        }
    ]
]

chat_messages = [tokenizer.apply_chat_template(m, tokenize=False, add_bos_token=False) for m in messages]
inputs = tokenizer(chat_messages, return_tensors="pt", padding=True, truncation=True, add_special_tokens=False).to("cuda")

output = llm_adapter.generate(inputs["input_ids"], max_new_tokens=100)
output["uncertainty_score"]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


odict_keys(['sequences', 'scores', 'attentions', 'hidden_states', 'past_key_values', 'full_attention_mask', 'context_lengths'])


/home/artemshelmanov/conda/lib/python3.13/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[[0.48153509069806605,
  0.70083100335193,
  0.37825823693552746,
  0.2932147494381937,
  0.926262639949862,
  0.7213100780196579]]

In [6]:
print("Model response and uncertainty scores:")
print(f'Response: {tokenizer.batch_decode(output["sequences"][:,len(inputs["input_ids"][0]):])}')
print(f'UE Scores: {output["uncertainty_score"][0]}')

Model response and uncertainty scores:
Response: ['Mercury is a logic programming language that was developed between 1986 and 1990 at the University of Edinburgh in Scotland. The first official release of Mercury was in 1993. So, the first appearance of Mercury was in 1993.</s>']
UE Scores: [0.48153509069806605, 0.70083100335193, 0.37825823693552746, 0.2932147494381937, 0.926262639949862, 0.7213100780196579]


In [7]:
def highlight_html_tokens(
    token_ids,
    positions_to_highlight,
    tokenizer,
    color="red",
    font_weight="bold"
):
    """
    Convert a list of token IDs into a readable string, highlight tokens at
    the specified positions in `positions_to_highlight`, and remove the leading
    '▁' that Mistral/Llama tokenizers use for word boundaries.
    
    Args:
        token_ids (List[int]): The sequence of token IDs.
        tokenizer: A Hugging Face tokenizer (e.g., for mistralai/Mistral-7B-Instruct-v0.2).
        positions_to_highlight (Set[int] or List[int]): 0-based indices of tokens to highlight.
        color (str): CSS color for the highlighted text (default "red").
        font_weight (str): CSS font weight (default "bold").
    
    Returns:
        str: An HTML string with some tokens highlighted.
    """
    # Convert the IDs to subword tokens (may contain leading "▁")
    raw_tokens = tokenizer.convert_ids_to_tokens(token_ids)
    
    # Ensure positions_to_highlight is a set for quick membership check
    if not isinstance(positions_to_highlight, set):
        positions_to_highlight = set(positions_to_highlight)
    
    final_pieces = []
    
    for idx, token in enumerate(raw_tokens):
        # If the token starts with "▁", replace that with a literal space
        if token.startswith("▁"):
            display_str = " " + token[1:]
        else:
            display_str = token
        
        # If this position is in positions_to_highlight, wrap in <span>
        if idx in positions_to_highlight:
            display_str = (
                f"<span style='color:{color}; font-weight:{font_weight};'>"
                f"{display_str}"
                "</span>"
            )
        
        final_pieces.append(display_str)
    
    # Join everything without extra spaces
    return "".join(final_pieces)

In [8]:
from IPython.display import HTML


def highlight_uncertain_claims(uncertainties, generated_tokens, claims):
    threshold = 0.5
    tokens_to_highlight = set()

    for ue_score, claim in zip(uncertainties, claims):
        if ue_score > threshold:
            tokens_to_highlight.update(claim)
    
    display(HTML(highlight_html_tokens(generated_tokens, tokens_to_highlight, tokenizer)))

In [9]:
highlight_uncertain_claims(
    output["uncertainty_score"][0],
    output["sequences"][:,len(inputs["input_ids"][0]):][0],
    [claim.aligned_token_ids for claim in output.deps["claims"][0]]
)